# Multi-Site Sweet Potato Variety Trial Statistical Analysis

**Author:** Sydney Seiter  
**Purpose:** Demonstration of field trial analysis for multi-site vegetable production research  
**Skills Demonstrated:**  
- Experimental design and statistical analysis
- Multi-site agricultural data synthesis
- Yield and quality metrics interpretation
- Publication-quality visualization
- Grower-focused recommendation development

---

## Research Context

This notebook analyzes data from a multi-site sweet potato variety trial conducted across three Southeastern production regions. The research question reflects real-world grower needs:

**Question:** Which sweet potato varieties perform best for yield and quality across diverse Southeastern growing conditions?

**Design:** Randomized complete block design (RCBD) with:
- 3 sites (Eastern NC, Aiken SC, Donalsonville GA)
- 4 varieties (Covington, Beauregard, Evangeline, Orleans)
- 4 replicate blocks per site
- 2 years of data (2024-2025)
- Yield, quality, and marketability metrics

This analysis demonstrates the statistical methods needed to support variety recommendations and communicate findings to growers and extension agents.

## Setup: Configure Output Directory

Choose where to save output files. Uncomment the Google Drive option if you want to save to Drive.

In [ ]:
import os

# OPTION 1: Save to current directory (default)
output_dir = '.'

# OPTION 2: Create a dedicated output folder
# output_dir = 'sweetpotato_trial_outputs'
# os.makedirs(output_dir, exist_ok=True)

# OPTION 3: Save to Google Drive (uncomment to use)
# from google.colab import drive
# drive.mount('/content/drive')
# output_dir = '/content/drive/MyDrive/SweetPotatoTrialAnalysis'
# os.makedirs(output_dir, exist_ok=True)

print(f"✓ Outputs will be saved to: {output_dir}")

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Install statsmodels if needed (uncomment for Colab)
# !pip install -q statsmodels

from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Set style for publication-quality plots
sns.set_style('whitegrid')
sns.set_context('talk')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

np.random.seed(42)
print("Libraries loaded successfully")

## 1. Generate Realistic Multi-Site Trial Data

Simulating 2 years of sweet potato variety trial data with realistic yield and quality metrics based on regional performance patterns.

In [ ]:
def generate_sweetpotato_trial_data():
    """
    Generate realistic multi-site sweet potato variety trial data.
    Variety performance and site characteristics based on industry standards.
    """
    
    sites = ['Eastern NC', 'Aiken SC', 'Donalsonville GA']
    varieties = ['Covington', 'Beauregard', 'Evangeline', 'Orleans']
    years = [2024, 2025]
    blocks = [1, 2, 3, 4]
    
    data_rows = []
    
    # Site-specific baseline characteristics (based on soil type, climate)
    site_characteristics = {
        'Eastern NC': {
            'base_yield': 450,  # cwt/acre (hundredweight per acre)
            'base_jumbo_pct': 35,  # % jumbo grade (>3.5" diameter)
            'base_no1_pct': 45,   # % #1 grade
            'base_cull_pct': 8,   # % culls
            'soil_type': 'Sandy loam',
            'avg_temp_f': 75,
            'disease_pressure': 'moderate'  # Affects quality
        },
        'Aiken SC': {
            'base_yield': 420,
            'base_jumbo_pct': 32,
            'base_no1_pct': 48,
            'base_cull_pct': 10,
            'soil_type': 'Sandy clay loam',
            'avg_temp_f': 77,
            'disease_pressure': 'moderate'
        },
        'Donalsonville GA': {
            'base_yield': 480,
            'base_jumbo_pct': 40,
            'base_no1_pct': 42,
            'base_cull_pct': 7,
            'soil_type': 'Loamy sand',
            'avg_temp_f': 79,
            'disease_pressure': 'low'  # Better drainage
        }
    }
    
    # Variety-specific characteristics (industry knowledge)
    variety_characteristics = {
        'Covington': {
            'yield_multiplier': 1.05,  # Industry standard, high yielder
            'jumbo_tendency': 1.1,     # Tends toward larger roots
            'disease_tolerance': 1.0,
            'notes': 'Industry standard, consistent performer'
        },
        'Beauregard': {
            'yield_multiplier': 0.95,  # Lower yield but good quality
            'jumbo_tendency': 0.9,
            'disease_tolerance': 0.85,  # More disease susceptible
            'notes': 'Traditional variety, declining acreage'
        },
        'Evangeline': {
            'yield_multiplier': 1.0,
            'jumbo_tendency': 1.05,
            'disease_tolerance': 1.1,   # Better disease resistance
            'notes': 'Good disease package, Louisiana-developed'
        },
        'Orleans': {
            'yield_multiplier': 0.98,
            'jumbo_tendency': 0.95,
            'disease_tolerance': 1.05,
            'notes': 'Consistent, good storage'
        }
    }
    
    plot_id = 1
    
    for site in sites:
        site_char = site_characteristics[site]
        
        for year in years:
            # Year effect (2025 had better growing conditions)
            year_effect = 1.08 if year == 2025 else 1.0
            
            for block in blocks:
                # Block effect (field spatial variability)
                block_effect = np.random.normal(1.0, 0.05)
                
                for variety in varieties:
                    var_char = variety_characteristics[variety]
                    
                    # Total yield (cwt/acre)
                    base_yield = site_char['base_yield'] * var_char['yield_multiplier']
                    total_yield = base_yield * year_effect * block_effect + np.random.normal(0, 25)
                    total_yield = max(200, total_yield)  # Minimum viable yield
                    
                    # Grade distribution (influenced by variety and disease pressure)
                    disease_factor = 1.0 if site_char['disease_pressure'] == 'low' else \
                                    0.95 if site_char['disease_pressure'] == 'moderate' else 0.90
                    
                    jumbo_pct = site_char['base_jumbo_pct'] * var_char['jumbo_tendency'] * \
                               disease_factor * var_char['disease_tolerance'] + np.random.normal(0, 3)
                    jumbo_pct = np.clip(jumbo_pct, 15, 55)
                    
                    # Culls increase with disease susceptibility
                    cull_pct = site_char['base_cull_pct'] / var_char['disease_tolerance'] + \
                              np.random.normal(0, 1.5)
                    cull_pct = np.clip(cull_pct, 3, 20)
                    
                    # #1 grade is the remainder after jumbo and culls
                    no1_pct = 100 - jumbo_pct - cull_pct
                    
                    # Calculate yields by grade
                    jumbo_yield = total_yield * (jumbo_pct / 100)
                    no1_yield = total_yield * (no1_pct / 100)
                    cull_yield = total_yield * (cull_pct / 100)
                    
                    # Marketable yield (Jumbo + #1)
                    marketable_yield = jumbo_yield + no1_yield
                    
                    # Root count (typical sweet potato planting)
                    plants_per_acre = 5500  # Standard spacing
                    roots_per_plant = (total_yield / 50) / plants_per_acre  # ~50 lbs per cwt
                    
                    data_rows.append({
                        'plot_id': f'{site[:2]}-{year}-B{block}-{variety[:4]}',
                        'site': site,
                        'year': year,
                        'block': block,
                        'variety': variety,
                        'total_yield_cwt_acre': round(total_yield, 1),
                        'jumbo_yield_cwt_acre': round(jumbo_yield, 1),
                        'no1_yield_cwt_acre': round(no1_yield, 1),
                        'cull_yield_cwt_acre': round(cull_yield, 1),
                        'marketable_yield_cwt_acre': round(marketable_yield, 1),
                        'jumbo_pct': round(jumbo_pct, 1),
                        'no1_pct': round(no1_pct, 1),
                        'cull_pct': round(cull_pct, 1),
                        'roots_per_plant': round(roots_per_plant, 2),
                        'planting_date': f'{year}-05-15',
                        'harvest_date': f'{year}-09-20'
                    })
    
    return pd.DataFrame(data_rows)

# Generate trial data
trial_data = generate_sweetpotato_trial_data()

print(f"Trial dataset generated: {len(trial_data)} observations")
print(f"\nExperimental design:")
print(f"  Sites: {trial_data['site'].nunique()} (Eastern NC, Aiken SC, Donalsonville GA)")
print(f"  Years: {trial_data['year'].nunique()} (2024-2025)")
print(f"  Varieties: {trial_data['variety'].nunique()}")
print(f"  Blocks per site: {trial_data['block'].nunique()}")
print(f"\nObservations per site-year-variety combination: {len(trial_data) // (3*2*4)}")
print("\nVarieties tested:")
for variety in trial_data['variety'].unique():
    print(f"  - {variety}")
print("\nFirst 10 rows:")
print(trial_data.head(10))

## 2. Exploratory Data Analysis

Visual exploration of variety performance across sites.

In [ ]:
# Create comprehensive visualization of variety performance
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Sweet Potato Variety Performance: Yield and Quality Across Sites', 
             fontsize=16, fontweight='bold', y=1.00)

metrics = [
    ('total_yield_cwt_acre', 'Total Yield (cwt/acre)', axes[0,0]),
    ('marketable_yield_cwt_acre', 'Marketable Yield (cwt/acre)', axes[0,1]),
    ('jumbo_pct', 'Jumbo Grade (%)', axes[0,2]),
    ('no1_pct', '#1 Grade (%)', axes[1,0]),
    ('cull_pct', 'Cull (%)', axes[1,1]),
    ('roots_per_plant', 'Roots per Plant', axes[1,2])
]

for metric, title, ax in metrics:
    sns.boxplot(data=trial_data, x='site', y=metric, hue='variety', ax=ax)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Site')
    ax.set_ylabel(title)
    ax.legend(title='Variety', loc='best', fontsize=8)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=15, ha='right')

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'sweetpotato_variety_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Variety comparison visualizations created")

## 3. Statistical Analysis: Two-Way ANOVA

Testing for variety and site effects on yield and quality metrics.

In [ ]:
def perform_anova(data, response_var, title):
    """
    Perform two-way ANOVA with variety, site, and their interaction.
    Returns ANOVA table and model.
    """
    # Fit model: response ~ variety + site + variety:site
    model = ols(f'{response_var} ~ C(variety) + C(site) + C(variety):C(site)', 
                data=data).fit()
    anova_table = anova_lm(model, typ=2)
    
    print(f"\n{'='*70}")
    print(f"ANOVA: {title}")
    print(f"{'='*70}")
    print(anova_table)
    
    # Interpret results
    alpha = 0.05
    print(f"\nInterpretation (α = {alpha}):")
    
    for effect in ['C(variety)', 'C(site)', 'C(variety):C(site)']:
        if effect in anova_table.index:
            p_value = anova_table.loc[effect, 'PR(>F)']
            f_value = anova_table.loc[effect, 'F']
            
            effect_name = effect.replace('C(', '').replace(')', '').replace(':', ' × ')
            
            if p_value < alpha:
                print(f"  {effect_name}: SIGNIFICANT (F={f_value:.2f}, p={p_value:.4f})")
            else:
                print(f"  {effect_name}: Not significant (F={f_value:.2f}, p={p_value:.4f})")
    
    # R-squared
    print(f"\nModel R²: {model.rsquared:.3f} (explains {model.rsquared*100:.1f}% of variance)")
    
    return anova_table, model

# Analyze key performance metrics
anova_results = {}

anova_results['total_yield'] = perform_anova(trial_data, 'total_yield_cwt_acre', 
                                              'Total Yield')

anova_results['marketable'] = perform_anova(trial_data, 'marketable_yield_cwt_acre', 
                                             'Marketable Yield')

anova_results['jumbo'] = perform_anova(trial_data, 'jumbo_pct', 
                                        'Jumbo Grade Percentage')

anova_results['culls'] = perform_anova(trial_data, 'cull_pct', 
                                        'Cull Percentage')

## 4. Post-Hoc Analysis: Variety Means Comparison

Tukey HSD test to identify significant pairwise differences between varieties.

In [ ]:
# Create variety-site combinations for detailed comparison
trial_data['variety_site'] = trial_data['variety'] + ' - ' + trial_data['site']

# Tukey HSD for total yield
print("\n" + "="*70)
print("TUKEY HSD POST-HOC TEST: Total Yield")
print("="*70)

tukey_yield = pairwise_tukeyhsd(endog=trial_data['total_yield_cwt_acre'],
                                groups=trial_data['variety'],
                                alpha=0.05)
print(tukey_yield)

# Summary statistics by variety and site
print("\n" + "="*70)
print("VARIETY PERFORMANCE BY SITE: Total Yield (cwt/acre)")
print("="*70)
yield_summary = trial_data.groupby(['site', 'variety'])['total_yield_cwt_acre'].agg(
    ['mean', 'std', 'count']
).round(1)
print(yield_summary)

print("\n" + "="*70)
print("VARIETY PERFORMANCE BY SITE: Marketable Yield (cwt/acre)")
print("="*70)
market_summary = trial_data.groupby(['site', 'variety'])['marketable_yield_cwt_acre'].agg(
    ['mean', 'std', 'count']
).round(1)
print(market_summary)

print("\n" + "="*70)
print("QUALITY METRICS BY VARIETY (Across all sites)")
print("="*70)
quality_summary = trial_data.groupby('variety')[['jumbo_pct', 'no1_pct', 'cull_pct']].agg(
    ['mean', 'std']
).round(1)
print(quality_summary)

## 5. Year-to-Year Consistency Analysis

Examining variety stability across years and sites.

In [ ]:
# Visualize year-to-year performance
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Sweet Potato Variety Stability: 2024 vs 2025 Performance',
             fontsize=16, fontweight='bold')

# Total yield by year
yearly_yield = trial_data.groupby(['year', 'variety'])['total_yield_cwt_acre'].mean().reset_index()
for variety in trial_data['variety'].unique():
    variety_data = yearly_yield[yearly_yield['variety'] == variety]
    axes[0].plot(variety_data['year'], variety_data['total_yield_cwt_acre'],
                marker='o', linewidth=3, label=variety)

axes[0].set_title('Total Yield by Year', fontweight='bold')
axes[0].set_ylabel('Total Yield (cwt/acre)')
axes[0].set_xlabel('Year')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks([2024, 2025])

# Marketable yield by year
yearly_market = trial_data.groupby(['year', 'variety'])['marketable_yield_cwt_acre'].mean().reset_index()
for variety in trial_data['variety'].unique():
    variety_data = yearly_market[yearly_market['variety'] == variety]
    axes[1].plot(variety_data['year'], variety_data['marketable_yield_cwt_acre'],
                marker='o', linewidth=3, label=variety)

axes[1].set_title('Marketable Yield by Year', fontweight='bold')
axes[1].set_ylabel('Marketable Yield (cwt/acre)')
axes[1].set_xlabel('Year')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks([2024, 2025])

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'year_to_year_stability.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Year-to-year stability analysis complete")

## 6. Grower Decision Tool: Yield × Quality Matrix

Creating a practical decision-making tool for variety selection.

In [ ]:
# Calculate variety performance scores
variety_scores = trial_data.groupby('variety').agg({
    'marketable_yield_cwt_acre': 'mean',
    'jumbo_pct': 'mean',
    'cull_pct': 'mean'
}).round(1)

# Create scatter plot: Marketable Yield vs Jumbo %
fig, ax = plt.subplots(figsize=(12, 8))

for variety in trial_data['variety'].unique():
    variety_data = trial_data[trial_data['variety'] == variety]
    ax.scatter(variety_data['marketable_yield_cwt_acre'], 
              variety_data['jumbo_pct'],
              s=150, alpha=0.6, label=variety)

# Add variety labels at mean positions
for variety in variety_scores.index:
    ax.annotate(variety, 
               (variety_scores.loc[variety, 'marketable_yield_cwt_acre'],
                variety_scores.loc[variety, 'jumbo_pct']),
               fontsize=12, fontweight='bold',
               xytext=(10, 10), textcoords='offset points')

# Add quadrant lines
mean_yield = trial_data['marketable_yield_cwt_acre'].mean()
mean_jumbo = trial_data['jumbo_pct'].mean()
ax.axvline(mean_yield, color='gray', linestyle='--', alpha=0.5, linewidth=2)
ax.axhline(mean_jumbo, color='gray', linestyle='--', alpha=0.5, linewidth=2)

# Add quadrant labels
ax.text(mean_yield + 20, mean_jumbo + 3, 'High Yield\nHigh Jumbo', 
       fontsize=10, style='italic', alpha=0.7, ha='center')
ax.text(mean_yield - 20, mean_jumbo + 3, 'Lower Yield\nHigh Jumbo', 
       fontsize=10, style='italic', alpha=0.7, ha='center')
ax.text(mean_yield + 20, mean_jumbo - 3, 'High Yield\nLower Jumbo', 
       fontsize=10, style='italic', alpha=0.7, ha='center')

ax.set_xlabel('Marketable Yield (cwt/acre)', fontsize=12, fontweight='bold')
ax.set_ylabel('Jumbo Grade (%)', fontsize=12, fontweight='bold')
ax.set_title('Variety Selection Matrix: Yield vs Premium Quality\n(Across all sites and years)',
            fontsize=14, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'variety_decision_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Grower decision tool created")

## 7. Extension Summary for Growers

Translating statistical findings into practical variety recommendations.

In [ ]:
# Generate grower-focused summary
print("\n" + "="*80)
print("SWEET POTATO VARIETY TRIAL SUMMARY: 2024-2025")
print("Sites: Eastern NC, Aiken SC, Donalsonville GA")
print("="*80)

print("\n📊 KEY FINDINGS:")
print("\n1. VARIETY PERFORMANCE (2-year average across all sites):")

overall_performance = trial_data.groupby('variety').agg({
    'total_yield_cwt_acre': 'mean',
    'marketable_yield_cwt_acre': 'mean',
    'jumbo_pct': 'mean',
    'no1_pct': 'mean',
    'cull_pct': 'mean'
}).round(1)

# Rank varieties by marketable yield
overall_performance = overall_performance.sort_values('marketable_yield_cwt_acre', ascending=False)

for i, (variety, row) in enumerate(overall_performance.iterrows(), 1):
    print(f"\n   {i}. {variety}:")
    print(f"      • Total Yield: {row['total_yield_cwt_acre']:.0f} cwt/acre")
    print(f"      • Marketable Yield: {row['marketable_yield_cwt_acre']:.0f} cwt/acre")
    print(f"      • Jumbo %: {row['jumbo_pct']:.1f}%  |  #1 %: {row['no1_pct']:.1f}%  |  Culls: {row['cull_pct']:.1f}%")

print("\n2. SITE-SPECIFIC OBSERVATIONS:")
site_performance = trial_data.groupby('site')['total_yield_cwt_acre'].agg(['mean', 'max']).round(0)
for site in site_performance.index:
    best_variety_site = trial_data[trial_data['site']==site].groupby('variety')['marketable_yield_cwt_acre'].mean().idxmax()
    avg_yield = site_performance.loc[site, 'mean']
    print(f"   • {site}: Avg {avg_yield:.0f} cwt/ac, top variety = {best_variety_site}")

print("\n3. YEAR EFFECTS:")
year_comparison = trial_data.groupby('year')['total_yield_cwt_acre'].mean()
year_diff = year_comparison[2025] - year_comparison[2024]
year_pct = (year_diff / year_comparison[2024]) * 100
print(f"   • 2024 avg: {year_comparison[2024]:.0f} cwt/acre")
print(f"   • 2025 avg: {year_comparison[2025]:.0f} cwt/acre")
print(f"   • Year-over-year change: {year_diff:+.0f} cwt/acre ({year_pct:+.1f}%)")
print(f"   • 2025 had better growing conditions")

print("\n4. QUALITY CONSIDERATIONS:")
# Best jumbo producer
best_jumbo = overall_performance['jumbo_pct'].idxmax()
best_jumbo_pct = overall_performance.loc[best_jumbo, 'jumbo_pct']
print(f"   • Highest jumbo %: {best_jumbo} ({best_jumbo_pct:.1f}%)")

# Lowest culls
lowest_cull = overall_performance['cull_pct'].idxmin()
lowest_cull_pct = overall_performance.loc[lowest_cull, 'cull_pct']
print(f"   • Lowest cull %: {lowest_cull} ({lowest_cull_pct:.1f}%)")

# Most consistent (lowest CV for marketable yield)
variety_cv = trial_data.groupby('variety')['marketable_yield_cwt_acre'].agg(
    lambda x: (x.std() / x.mean()) * 100
)
most_stable = variety_cv.idxmin()
print(f"   • Most consistent performer: {most_stable} (lowest yield variability)")

print("\n✅ RECOMMENDATIONS:")
print("   1. OVERALL WINNER: Choose varieties ranking #1-2 for marketable yield")
print("   2. FOR PREMIUM MARKETS: Select variety with highest jumbo %")
print("   3. FOR RISK MANAGEMENT: Consider most consistent variety for stable returns")
print("   4. SITE-SPECIFIC: Match variety to your local conditions based on site data")
print("   5. ALL VARIETIES TESTED: Commercially viable across Southeast region")

print("\n💰 ECONOMIC NOTES:")
print("   • Jumbo grade typically brings premium pricing over #1 grade")
print("   • Marketable yield more important than total yield for profitability")
print("   • Cull percentage directly impacts packout and profitability")
print("   • Consider storage characteristics for your marketing timeline")

print("\n" + "="*80)
print("Full statistical details available in ANOVA output above")
print("="*80)

## Summary

This analysis demonstrates:

✅ **Field trial design expertise** - Multi-site RCBD with commercial varieties  
✅ **Statistical rigor** - ANOVA, post-hoc tests, variety rankings  
✅ **Vegetable production knowledge** - Realistic yield/quality metrics and grading standards  
✅ **Multi-site synthesis** - Performance patterns across Southeastern production regions  
✅ **Grower communication** - Translating statistics into practical variety recommendations  
✅ **Economic perspective** - Marketable yield and quality metrics for profitability  
✅ **Publication-quality visualization** - Professional graphics for extension/industry  

**Skills directly applicable to agricultural research:**
- Analyzing data from multi-location variety trials
- Supporting statistical analysis for grower publications
- Creating decision tools for variety selection
- Synthesizing findings across diverse growing environments
- Communicating results to non-statistician stakeholders (growers, extension)
- Understanding commercial vegetable production economics

In [ ]:
# Save processed data
trial_data.to_csv(os.path.join(output_dir, 'sweetpotato_trial_results.csv'), index=False)
print("\n✅ Analysis complete. Results saved to sweetpotato_trial_results.csv")